# Studio Project 1 — Fine-Tuning ANN (from scratch) pada CIFAR-10
**Mata Kuliah:** Pembelajaran Mendalam (IF25-40401)

**Tujuan:** Membangun arsitektur *Artificial Neural Network* (MLP, bukan CNN) *from scratch* dengan PyTorch,
melakukan eksperimen sistematis (optimizer, scheduler, regularisasi, kapasitas model, early stopping,
checkpointing), lalu menyimpulkan strategi pelatihan terbaik berbasis bukti (*evidence-based*).

> **Catatan penting:** Sesuai spesifikasi tugas, model **WAJIB berupa MLP/ANN standar** (input di-*flatten*
> menjadi vektor 32×32×3 = 3072), **bukan CNN**. Semua modul di bawah ini konsisten dengan batasan tersebut.

**Cara pakai notebook ini:**
1. Jalankan di **Google Colab** atau **Kaggle Notebook** (bukan di sandbox lokal) agar dataset CIFAR-10 bisa
   diunduh dan GPU tersedia.
2. Colab: `Runtime > Change runtime type > T4 GPU`.
3. Jalankan sel dari atas ke bawah (`Run All`). Total waktu training penuh (semua skenario) diperkirakan
   30–60 menit dengan GPU T4.
4. Setelah selesai dijalankan penuh (kode + output grafik ikut ter-*render*), simpan/unduh sebagai PDF
   sesuai format nama file: `STUDIO1_Kelompok[X]_[NamaAnggota1]_[NamaAnggota2].pdf`.


## 0. Setup & Import Library

Kita gunakan PyTorch + torchvision untuk data, `sklearn` untuk metrik evaluasi (confusion matrix),
dan `matplotlib`/`seaborn` untuk visualisasi.

In [2]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Menggunakan device:", device)
if device.type == "cpu":
    print("PERINGATAN: GPU tidak terdeteksi. Training MLP di CPU tetap memungkinkan namun lebih lambat.")


Menggunakan device: cpu
PERINGATAN: GPU tidak terdeteksi. Training MLP di CPU tetap memungkinkan namun lebih lambat.


## 1. Preparation & Preprocessing

Langkah:
1. Unduh CIFAR-10 lewat `torchvision.datasets.CIFAR10`.
2. Normalisasi menggunakan mean/std standar CIFAR-10.
3. **Augmentasi dasar** (RandomCrop + padding, RandomHorizontalFlip) — **hanya pada data latih**, bukan
   validasi/test, supaya evaluasi tetap objektif.
4. **Flatten** gambar 3×32×32 menjadi vektor 3072, karena arsitektur yang dipakai adalah MLP standar (bukan CNN).
5. Split data latih asli (50.000 gambar) menjadi **45.000 train / 5.000 validation**. Data validasi memakai
   transform tanpa augmentasi (dua instance dataset terpisah dengan transform berbeda, disatukan lewat index
   yang sama, supaya augmentasi tidak "bocor" ke data validasi).

In [ ]:
import torch
print   ("CUDA Tersedia:", torch.cuda.is_available())
print("Versi CUDA PyTorch:", torch.version.cuda)
print("Nama GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tidak ada GPU")

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    transforms.Lambda(lambda x: torch.flatten(x)),   # flatten -> 3072
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    transforms.Lambda(lambda x: torch.flatten(x)),
])

DATA_ROOT = "./data"

# Dua instance dataset training dengan transform berbeda (augmented vs plain),
# supaya kita bisa pisahkan subset train (augmented) dan val (plain) dari index yang sama.
train_full_aug   = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=train_transform)
train_full_plain = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=eval_transform)
test_set          = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=eval_transform)

CLASS_NAMES = train_full_aug.classes
print("Kelas:", CLASS_NAMES)

# Split train/val 45000/5000 dengan seed tetap
indices = list(range(len(train_full_aug)))
rng = np.random.RandomState(SEED)
rng.shuffle(indices)

VAL_SIZE = 5000
val_idx, train_idx = indices[:VAL_SIZE], indices[VAL_SIZE:]

train_set = Subset(train_full_aug, train_idx)     # dengan augmentasi
val_set   = Subset(train_full_plain, val_idx)      # tanpa augmentasi

print(f"Jumlah data -> train: {len(train_set)}, val: {len(val_set)}, test: {len(test_set)}")

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


0.6%

In [ ]:
# Sanity check: tampilkan beberapa contoh gambar (sebelum flatten, untuk visual saja)
def unnormalize(img_tensor):
    mean = torch.tensor(CIFAR10_MEAN).view(3,1,1)
    std = torch.tensor(CIFAR10_STD).view(3,1,1)
    return img_tensor * std + mean

preview_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
preview_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=preview_transform)

fig, axes = plt.subplots(1, 8, figsize=(16,2))
for i, ax in enumerate(axes):
    img, label = preview_set[i]
    img = unnormalize(img).permute(1,2,0).clamp(0,1).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.suptitle("Contoh data CIFAR-10 (sebelum di-flatten untuk MLP)")
plt.tight_layout()
plt.show()


## 2. Arsitektur Baseline Model (MLP / ANN, from scratch)

Arsitektur dibuat **fleksibel** (jumlah hidden layer, jumlah neuron, dropout, batch normalization semua bisa
diatur lewat parameter) agar bisa dipakai ulang di seluruh skenario eksperimen (butir C.3 spesifikasi tugas),
tanpa menulis ulang class untuk tiap eksperimen.

**Baseline:** 1 Input Layer (3072) → 2 Hidden Layer (512, 256) → 1 Output Layer (10), aktivasi ReLU,
tanpa dropout/batchnorm.

In [ ]:
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (ANN) from scratch untuk klasifikasi CIFAR-10.
    Input: vektor flatten berukuran 3072 (32*32*3)
    Output: 10 logits (10 kelas CIFAR-10)
    """
    def __init__(self, input_size=3072, hidden_sizes=(512, 256), num_classes=10,
                 use_dropout=False, dropout_p=0.5, use_batchnorm=False):
        super().__init__()
        layers = []
        in_dim = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            if use_dropout:
                layers.append(nn.Dropout(p=dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Quick check arsitektur baseline
baseline_check = MLP(hidden_sizes=(512, 256))
print(baseline_check)
n_params = sum(p.numel() for p in baseline_check.parameters() if p.requires_grad)
print(f"Total parameter trainable: {n_params:,}")


## 3. Early Stopping & Model Checkpointing (from scratch)

Diimplementasikan manual (tanpa library eksternal) sesuai butir C.4 spesifikasi:
- Memantau **validation loss** setiap epoch.
- Menyimpan bobot model (`torch.save`) setiap kali validation loss membaik (checkpoint = model terbaik).
- Menghentikan pelatihan (*early stop*) jika tidak ada perbaikan selama `patience` epoch berturut-turut.

In [ ]:
class EarlyStopping:
    """
    Early stopping + model checkpointing manual berbasis validation loss.
    """
    def __init__(self, patience=5, min_delta=1e-4, checkpoint_path="checkpoint.pt", verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path
        self.verbose = verbose

        self.best_loss = np.inf
        self.counter = 0
        self.early_stop = False
        self.best_epoch = -1

    def step(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            # Ada perbaikan -> simpan checkpoint, reset counter
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            if self.verbose:
                print(f"  -> Val loss membaik ({val_loss:.4f}). Checkpoint disimpan ke '{self.checkpoint_path}'.")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  -> Val loss tidak membaik ({self.counter}/{self.patience}).")
            if self.counter >= self.patience:
                self.early_stop = True


## 4. Training & Evaluation Loop

Fungsi generik `train_model` dipakai untuk semua skenario eksperimen agar perbandingan antar skenario *fair*
(hanya komponen yang diuji yang berbeda, sisanya identik).

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Satu epoch training (jika optimizer diberikan) atau evaluasi (jika optimizer=None)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if is_train:
                loss.backward()
                optimizer.step()
            running_loss += loss.item() * x.size(0)
            correct += (out.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, scheduler=None,
                 epochs=30, patience=5, tag="exp", verbose=True):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    stopper = EarlyStopping(patience=patience, checkpoint_path=f"{tag}_best.pt", verbose=False)

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        if scheduler is not None:
            if isinstance(scheduler, ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        if verbose:
            print(f"[{tag}] Epoch {epoch:02d}/{epochs} | "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={current_lr:.6f}")

        stopper.step(val_loss, model, epoch)
        if stopper.early_stop:
            print(f"[{tag}] Early stopping dipicu pada epoch {epoch} "
                  f"(best val_loss={stopper.best_loss:.4f} @ epoch {stopper.best_epoch}).")
            break

    elapsed = time.time() - t0
    # Muat kembali bobot TERBAIK (checkpoint), bukan bobot epoch terakhir
    model.load_state_dict(torch.load(f"{tag}_best.pt"))

    history["training_time_sec"] = elapsed
    history["best_epoch"] = stopper.best_epoch
    history["best_val_loss"] = stopper.best_loss
    history["best_val_acc"] = max(history["val_acc"]) if history["val_acc"] else None
    history["stopped_epoch"] = epoch
    return model, history


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train Acc")
    axes[1].plot(history["val_acc"], label="Val Acc")
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()

    plt.tight_layout()
    plt.show()


## 5. Training Baseline Model

Baseline: Adam (lr=1e-3) tanpa scheduler, tanpa dropout/batchnorm, arsitektur (512, 256).
Ini jadi titik acuan (*reference point*) sebelum eksperimen sistematis di bagian selanjutnya.

In [ ]:
baseline_model = MLP(hidden_sizes=(512, 256))
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=1e-3)

baseline_model, baseline_hist = train_model(
    baseline_model, train_loader, val_loader,
    optimizer=baseline_optimizer, scheduler=None,
    epochs=25, patience=5, tag="baseline"
)

plot_history(baseline_hist, "Baseline (Adam, no reg, 512-256)")
print(f"Baseline best val_acc: {baseline_hist['best_val_acc']:.4f} | waktu: {baseline_hist['training_time_sec']:.1f}s")


## 6. Skenario Eksperimen A — Optimizer & Scheduler

Membandingkan:
1. **Adam** (lr=1e-3) + **StepLR** (turunkan LR ×0.5 setiap 10 epoch)
2. **SGD + Momentum** (lr=1e-2, momentum=0.9) + **ReduceLROnPlateau** (turunkan LR jika val_loss stagnan)

Arsitektur dan seluruh pengaturan lain dibuat identik dengan baseline supaya perbedaan hasil murni berasal
dari optimizer & scheduler.

In [ ]:
results_a = {}

# --- A1: Adam + StepLR ---
model_a1 = MLP(hidden_sizes=(512, 256))
opt_a1 = optim.Adam(model_a1.parameters(), lr=1e-3)
sched_a1 = StepLR(opt_a1, step_size=10, gamma=0.5)
model_a1, hist_a1 = train_model(model_a1, train_loader, val_loader, opt_a1, sched_a1,
                                 epochs=25, patience=5, tag="expA_adam_steplr")
plot_history(hist_a1, "Exp A1: Adam + StepLR")
results_a["Adam+StepLR"] = hist_a1


In [ ]:
# --- A2: SGD + Momentum + ReduceLROnPlateau ---
model_a2 = MLP(hidden_sizes=(512, 256))
opt_a2 = optim.SGD(model_a2.parameters(), lr=1e-2, momentum=0.9)
sched_a2 = ReduceLROnPlateau(opt_a2, mode="min", factor=0.5, patience=3)
model_a2, hist_a2 = train_model(model_a2, train_loader, val_loader, opt_a2, sched_a2,
                                 epochs=25, patience=5, tag="expA_sgd_plateau")
plot_history(hist_a2, "Exp A2: SGD+Momentum + ReduceLROnPlateau")
results_a["SGD+Momentum+Plateau"] = hist_a2


In [ ]:
summary_a = pd.DataFrame({
    name: {
        "Best Val Acc": h["best_val_acc"],
        "Best Val Loss": h["best_val_loss"],
        "Best Epoch": h["best_epoch"],
        "Stopped Epoch": h["stopped_epoch"],
        "Training Time (s)": round(h["training_time_sec"], 1),
    }
    for name, h in results_a.items()
}).T
summary_a


**Interpretasi (isi otomatis berdasar angka di atas saat notebook dijalankan):** Optimizer dengan
`Best Val Acc` tertinggi dan konvergensi (mencapai `Best Epoch`) lebih cepat dipilih sebagai kandidat
optimizer/scheduler terbaik untuk dipakai di eksperimen berikutnya (variabel `BEST_OPTIMIZER_NAME` di sel
berikut).

In [ ]:
# Tentukan optimizer/scheduler terbaik berdasarkan val_acc tertinggi -> dipakai di eksperimen B & C
BEST_OPTIMIZER_NAME = summary_a["Best Val Acc"].astype(float).idxmax()
print("Optimizer & scheduler terbaik dari Eksperimen A:", BEST_OPTIMIZER_NAME)

def make_best_optimizer(params):
    """Mengembalikan (optimizer, scheduler) sesuai hasil terbaik Eksperimen A."""
    if BEST_OPTIMIZER_NAME == "Adam+StepLR":
        opt = optim.Adam(params, lr=1e-3)
        sch = StepLR(opt, step_size=10, gamma=0.5)
    else:
        opt = optim.SGD(params, lr=1e-2, momentum=0.9)
        sch = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    return opt, sch


## 7. Skenario Eksperimen B — Regularisasi (Dropout & Batch Normalization)

Menguji dampak Dropout dan Batch Normalization terhadap overfitting, menggunakan optimizer/scheduler
terbaik dari Eksperimen A. Arsitektur (512, 256) dipertahankan agar perbandingan adil.

Empat varian:
1. Tanpa regularisasi (kontrol/baseline arsitektur)
2. Dropout (p=0.5) saja
3. Batch Normalization saja
4. Dropout + Batch Normalization (kombinasi)

In [ ]:
results_b = {}
reg_variants = {
    "No-Reg":        dict(use_dropout=False, use_batchnorm=False),
    "Dropout-0.5":   dict(use_dropout=True,  dropout_p=0.5, use_batchnorm=False),
    "BatchNorm":     dict(use_dropout=False, use_batchnorm=True),
    "Dropout+BN":    dict(use_dropout=True,  dropout_p=0.3, use_batchnorm=True),
}

for name, kwargs in reg_variants.items():
    m = MLP(hidden_sizes=(512, 256), **kwargs)
    opt, sch = make_best_optimizer(m.parameters())
    m, hist = train_model(m, train_loader, val_loader, opt, sch,
                           epochs=25, patience=5, tag=f"expB_{name}")
    plot_history(hist, f"Exp B: {name}")
    results_b[name] = hist


In [ ]:
summary_b = pd.DataFrame({
    name: {
        "Best Val Acc": h["best_val_acc"],
        "Best Val Loss": h["best_val_loss"],
        "Final Train-Val Acc Gap": h["train_acc"][h["best_epoch"]-1] - h["val_acc"][h["best_epoch"]-1],
        "Best Epoch": h["best_epoch"],
        "Training Time (s)": round(h["training_time_sec"], 1),
    }
    for name, h in results_b.items()
}).T
summary_b


**Interpretasi:** `Final Train-Val Acc Gap` yang lebih kecil menandakan overfitting lebih terkendali.
Varian dengan `Best Val Acc` tertinggi **dan** gap yang wajar (tidak terlalu besar) dipilih sebagai
konfigurasi regularisasi terbaik untuk eksperimen kapasitas model berikutnya.

In [ ]:
BEST_REG_NAME = summary_b["Best Val Acc"].astype(float).idxmax()
BEST_REG_KWARGS = reg_variants[BEST_REG_NAME]
print("Konfigurasi regularisasi terbaik dari Eksperimen B:", BEST_REG_NAME, BEST_REG_KWARGS)


## 8. Skenario Eksperimen C — Kapasitas Model (Jumlah Hidden Layer / Neuron)

Menguji dampak penambahan kapasitas model terhadap akurasi akhir **dan** *training time*, dengan
optimizer/scheduler + regularisasi terbaik dari eksperimen sebelumnya tetap dipakai (fair comparison).

In [ ]:
results_c = {}
capacity_variants = {
    "Small (256-128)":        (256, 128),
    "Base (512-256)":         (512, 256),
    "Large (1024-512-256)":   (1024, 512, 256),
    "XLarge (2048-1024-512-256)": (2048, 1024, 512, 256),
}

for name, hidden_sizes in capacity_variants.items():
    m = MLP(hidden_sizes=hidden_sizes, **BEST_REG_KWARGS)
    opt, sch = make_best_optimizer(m.parameters())
    m, hist = train_model(m, train_loader, val_loader, opt, sch,
                           epochs=25, patience=5, tag=f"expC_{name.split()[0]}")
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    hist["n_params"] = n_params
    plot_history(hist, f"Exp C: {name}")
    results_c[name] = hist


In [ ]:
summary_c = pd.DataFrame({
    name: {
        "Hidden Sizes": str(capacity_variants[name]),
        "Params": f"{h['n_params']:,}",
        "Best Val Acc": h["best_val_acc"],
        "Best Val Loss": h["best_val_loss"],
        "Training Time (s)": round(h["training_time_sec"], 1),
    }
    for name, h in results_c.items()
}).T
summary_c


**Interpretasi:** Model dengan kapasitas lebih besar *biasanya* mencapai akurasi lebih tinggi, tetapi
dengan trade-off waktu pelatihan yang jauh lebih lama dan risiko overfitting lebih tinggi. Strategi "paling
efisien" (butir F objectives) dipilih dengan mempertimbangkan **rasio kenaikan akurasi terhadap kenaikan
waktu training**, bukan sekadar akurasi tertinggi.

In [ ]:
BEST_CAPACITY_NAME = summary_c["Best Val Acc"].astype(float).idxmax()
BEST_HIDDEN_SIZES = capacity_variants[BEST_CAPACITY_NAME]
print("Kapasitas terbaik (val_acc tertinggi) dari Eksperimen C:", BEST_CAPACITY_NAME, BEST_HIDDEN_SIZES)


## 9. Model Final (Kombinasi Konfigurasi Terbaik)

Melatih ulang dengan kombinasi terbaik dari ketiga eksperimen (optimizer/scheduler, regularisasi, kapasitas),
dengan epoch lebih banyak dan early stopping/checkpointing tetap aktif, agar model benar-benar konvergen.

In [ ]:
final_model = MLP(hidden_sizes=BEST_HIDDEN_SIZES, **BEST_REG_KWARGS)
final_opt, final_sched = make_best_optimizer(final_model.parameters())

final_model, final_hist = train_model(
    final_model, train_loader, val_loader,
    optimizer=final_opt, scheduler=final_sched,
    epochs=60, patience=7, tag="final_best_model"
)

plot_history(final_hist, "Model Final (Konfigurasi Terbaik)")
print(f"Model final -> Best Val Acc: {final_hist['best_val_acc']:.4f} "
      f"| Best Val Loss: {final_hist['best_val_loss']:.4f} "
      f"| Waktu training: {final_hist['training_time_sec']:.1f}s")


## 10. Evaluasi Akhir pada Test Set

Evaluasi model final (bobot checkpoint terbaik) pada **test set** CIFAR-10 (10.000 gambar, belum pernah
dilihat model): akurasi, classification report per kelas, dan confusion matrix.

In [ ]:
def evaluate_on_test(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            preds = out.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = evaluate_on_test(final_model, test_loader)
test_acc = accuracy_score(y_true, y_pred)
print(f"Akurasi Test Set (Model Final): {test_acc:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix — Model Final (Test Acc: {test_acc:.4f})")
plt.tight_layout()
plt.show()


In [ ]:
# Perbandingan akhir Train Loss vs Val Loss untuk model final (butir C.5)
plt.figure(figsize=(7,4))
plt.plot(final_hist["train_loss"], label="Train Loss")
plt.plot(final_hist["val_loss"], label="Val Loss")
plt.axvline(final_hist["best_epoch"]-1, color="green", linestyle="--", label="Best Checkpoint")
plt.title("Model Final — Train Loss vs Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.show()


## 11. Rangkuman Seluruh Eksperimen

In [ ]:
all_summary = pd.concat([
    summary_a.assign(Kategori="A: Optimizer/Scheduler"),
    summary_b.assign(Kategori="B: Regularisasi"),
    summary_c.assign(Kategori="C: Kapasitas Model"),
], axis=0)
all_summary


## 12. Kesimpulan & Strategi Pelatihan Terbaik (Evidence-Based)

> **[Isi bagian ini setelah notebook dijalankan penuh, berdasarkan angka aktual di tabel-tabel di atas.]**
> Template argumentasi yang bisa dipakai (sesuaikan dengan angka hasil run Anda):

1. **Optimizer & Scheduler:** `{BEST_OPTIMIZER_NAME}` memberikan validation accuracy tertinggi pada
   Eksperimen A karena … (jelaskan: konvergensi lebih stabil / lebih cepat mencapai loss rendah / dsb.,
   merujuk ke `summary_a`).
2. **Regularisasi:** `{BEST_REG_NAME}` paling efektif menekan overfitting, ditunjukkan oleh gap
   train-val accuracy yang lebih kecil pada `summary_b` dibanding varian *No-Reg*.
3. **Kapasitas Model:** `{BEST_CAPACITY_NAME}` memberi kenaikan akurasi terbaik, namun perlu dibandingkan
   dengan kenaikan *training time* pada `summary_c` — jika kenaikan akurasi marginal namun waktu training
   naik signifikan, model yang lebih kecil bisa dianggap "lebih efisien".
4. **Early Stopping & Checkpointing** berhasil mencegah model overfit lebih jauh dengan menghentikan
   training otomatis begitu validation loss berhenti membaik selama `patience` epoch, dan memastikan bobot
   yang dipakai untuk evaluasi akhir adalah bobot **terbaik**, bukan bobot epoch terakhir yang mungkin sudah
   overfit.
5. **Rekomendasi Strategi Terbaik:** kombinasi `[optimizer/scheduler terbaik] + [regularisasi terbaik] +
   [kapasitas model terbaik]` seperti dipakai pada **Model Final** di atas, dengan test accuracy akhir
   sebesar `{{test_acc:.4f}}` pada 10.000 gambar test set CIFAR-10 yang belum pernah dilihat model.

**Catatan keterbatasan MLP untuk data citra:** karena arsitektur ini adalah MLP (bukan CNN), model tidak
memiliki *inductive bias* untuk mempertahankan struktur spasial gambar (setiap piksel diperlakukan sebagai
fitur independen setelah *flatten*). Ini wajar menjadi salah satu penjelasan mengapa akurasi MLP pada
CIFAR-10 secara umum jauh lebih rendah dibanding CNN — poin ini bisa disampaikan saat sesi presentasi/diskusi
sebagai bagian dari analisis kritis, bukan sebagai kekurangan implementasi.
